In [2]:
import pandas as pd

In [3]:
ticker = "AAPL"


open_time = pd.to_datetime('2024-04-01')
close_time = pd.to_datetime('2024-09-01')

df = pd.read_csv(f"full_texts/{ticker}_full_texts_2024-04-01-2024-09-01.csv")

texts = df.news.values

In [4]:
texts[0]

'Apple Inc. AAPL reportedly plans to introduce a new polished titanium finish for its upcoming iPhone 16 Pro models.\nWhat Happened: A new leak suggests that Apple’s iPhone 16 Pro will feature a polished titanium finish. The leak, from a source known as “yeux1122,” states that the new manufacturing process will give the iPhone 16 Pro a more polished look than its predecessor, the iPhone 15 Pro, reported AppleInsider. \nThe leak also suggests that the new production method will result in a different finish for each color variant of the iPhone 16 Pro. The new titanium chassis is expected to be more polished and possibly more scratch-resistant than the current brushed titanium look of the iPhone 15 Pro.\nSubscribe to the\xa0Benzinga Tech Trends newsletter\xa0to get all the latest tech developments delivered to your inbox.\nHowever, the leaker’s track record with hardware reports is hit or miss, and the changes could be unrelated to the titanium\xa0finish, such as production line\xa0improv

In [5]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

# Carregar o tokenizer e o modelo FinBERT pré-treinado para classificação de sentimento
tokenizer = BertTokenizer.from_pretrained('yiyanghkust/finbert-tone')
model = BertForSequenceClassification.from_pretrained('yiyanghkust/finbert-tone')

# Colocar o modelo em modo de avaliação
model.eval()


/home/guicbelon/Documentos/IC/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30873, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [11]:
print(texts[0])

Apple Inc. AAPL reportedly plans to introduce a new polished titanium finish for its upcoming iPhone 16 Pro models.
What Happened: A new leak suggests that Apple’s iPhone 16 Pro will feature a polished titanium finish. The leak, from a source known as “yeux1122,” states that the new manufacturing process will give the iPhone 16 Pro a more polished look than its predecessor, the iPhone 15 Pro, reported AppleInsider. 
The leak also suggests that the new production method will result in a different finish for each color variant of the iPhone 16 Pro. The new titanium chassis is expected to be more polished and possibly more scratch-resistant than the current brushed titanium look of the iPhone 15 Pro.
Subscribe to the Benzinga Tech Trends newsletter to get all the latest tech developments delivered to your inbox.
However, the leaker’s track record with hardware reports is hit or miss, and the changes could be unrelated to the titanium finish, such as production line improvements for speed 

In [9]:
def classify_sentiment(text):
    # Tokenizar o texto
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    
    # Realizar inferência com o FinBERT
    with torch.no_grad():
        outputs = model(**inputs)
    
    # A saída é o logits (pontuações brutas) para cada classe
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()
    print(logits)
    print(predicted_class)

    # Classes de sentimento do FinBERT (0: Negativo, 1: Neutro, 2: Positivo)
    sentiment_labels = {0: "Negativo", 1: "Neutro", 2: "Positivo"}
    
    return sentiment_labels[predicted_class]

# Testar com um texto de exemplo
texto_financeiro = texts[0]
resultado = classify_sentiment(texto_financeiro)
print(f"O sentimento do texto é: {resultado}")


tensor([[ 5.2690, -3.3498, -4.4964]])
0
O sentimento do texto é: Negativo


In [7]:
# Função para resumir o texto com BART
def resumir_texto_bart(texto, max_length=150):
    inputs = tokenizer.encode("summarize: " + texto, return_tensors="pt", max_length=1024, truncation=True)
    resumo_ids = model.generate(inputs, max_length=max_length, min_length=30, length_penalty=2.0, num_beams=4, early_stopping=True)
    resumo = tokenizer.decode(resumo_ids[0], skip_special_tokens=True)
    return resumo


resumir_texto_bart(texto_financeiro)

TypeError: The current model class (BertForSequenceClassification) is not compatible with `.generate()`, as it doesn't have a language model head. Classes that support generation often end in one of these names: ['ForCausalLM', 'ForConditionalGeneration', 'ForSpeechSeq2Seq', 'ForVision2Seq'].

In [1]:
from transformers import BertModel, BertTokenizer
import torch


/home/guicbelon/Documentos/IC/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Carregar modelo e tokenizer
model = BertModel.from_pretrained('bert-base-uncased')

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [17]:
# Tokenizar a frase
sentence = texts[0]
inputs = tokenizer(sentence, return_tensors='pt')
inputs

{'input_ids': tensor([[  101,  6207,  4297,  1012,  9779, 24759,  7283,  3488,  2000,  8970,
          1037,  2047, 12853, 23431,  3926,  2005,  2049,  9046, 18059,  2385,
          4013,  4275,  1012,  2054,  3047,  1024,  1037,  2047, 17271,  6083,
          2008,  6207,  1521,  1055, 18059,  2385,  4013,  2097,  3444,  1037,
         12853, 23431,  3926,  1012,  1996, 17271,  1010,  2013,  1037,  3120,
          2124,  2004,  1523,  6300,  5602, 14526, 19317,  1010,  1524,  2163,
          2008,  1996,  2047,  5814,  2832,  2097,  2507,  1996, 18059,  2385,
          4013,  1037,  2062, 12853,  2298,  2084,  2049,  8646,  1010,  1996,
         18059,  2321,  4013,  1010,  2988,  6207,  7076, 18688,  1012,  1996,
         17271,  2036,  6083,  2008,  1996,  2047,  2537,  4118,  2097,  2765,
          1999,  1037,  2367,  3926,  2005,  2169,  3609,  8349,  1997,  1996,
         18059,  2385,  4013,  1012,  1996,  2047, 23431, 11832,  2003,  3517,
          2000,  2022,  2062, 12853,  

In [18]:
# Gera o embedding com o BERT
outputs = model(**inputs)

In [19]:
sentence_embedding = outputs.last_hidden_state.mean(dim=1).detach().numpy()

sentence_embedding.shape

(1, 768)

In [20]:
from transformers import BertModel, BertTokenizer
import torch

# Carregar o modelo BERT-large e o tokenizer
model = BertModel.from_pretrained('bert-large-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-large-uncased')


In [22]:

# Texto de exemplo
sentence = "Este é um exemplo de texto para gerar embeddings com BERT-large."

# Tokenizar o texto e converter para tensores
inputs = tokenizer(sentence, return_tensors='pt')

# Colocar o modelo em modo de avaliação (inference mode)
model.eval()

# Gerar os embeddings com o BERT-large
with torch.no_grad():  # Não queremos calcular gradientes aqui
    outputs = model(**inputs)

# O embedding final da frase é a média dos embeddings de todas as palavras na última camada
sentence_embedding = outputs.last_hidden_state.mean(dim=1).squeeze()

# Converter para numpy array (opcional)
sentence_embedding = sentence_embedding.detach().numpy()

# Ver a dimensão do embedding
print(f"Dimensão do embedding: {sentence_embedding.shape}")
print(sentence_embedding)  # Exibir o vetor gerado


Dimensão do embedding: (1024,)
[ 0.00601423 -0.1947694   0.5819392  ... -0.25224164 -0.15248394
 -0.50471574]
